In [1]:
# conda env 'misr'
"""This notebook extracts cloud motion information 
from a MISR hdf file. Its main purpose is to convert the hdf to 
a netCDF file to eliminate the need for using the MisrToolkit 
in downstream scripts.

This notebook is very similar to MISR_preprocess1.ipynb

Generates a MISR_CMV*.nc file. The MISR_CMV file is not an input 
to run_at3d.py. Instead, the cloud motion vector is extracted and 
from this data and supplied directly to run_at3d.py
"""
import MisrToolkit as mtk
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

from pathlib import Path
PROJECT_ROOT = Path().resolve().parent.parent
RAW_DATA = PROJECT_ROOT / 'data' / 'raw'
INTERIM_DATA = PROJECT_ROOT / 'data' / 'interim'

In [2]:
def nearest_sample(lat, lon, path, resolution):
    region = mtk.MtkRegion(lat,lon, 1, 1, "km")
    lats, lons = region.snap_to_grid(path, resolution).create_latlon()
    lats = lats.ravel()
    lons = lons.ravel()
    lat_diff = lats - lat
    lon_diff = lons - lon
    D = (lat_diff**2 + lon_diff**2)**0.5
    return lats[np.argmin(D)], lons[np.argmin(D)]

In [3]:
out_file = INTERIM_DATA / 'MISR_CMV_CAMP2Ex_RF17.nc'

database = RAW_DATA / 'MISR'
tc_cloud_general = 'MISR_AM1_TC_CLOUD_P{:d}_O{:d}_F01_0001.hdf'
path = 117
orbit = 105254
close_lat = 19.266
close_lon = 119.724
lat, lon = nearest_sample(close_lat, close_lon, path, 17600)
n = 7
# region = mtk.MtkRegion(path, 74, 76) # RF17
region = mtk.MtkRegion(lat, lon, 8.8*n, 8.8*n, "km")
lat17600, lon17600 = region.snap_to_grid(path, 17600).create_latlon()
# assert lat17600.shape[0] % 2 == 1
# assert lon17600.shape[0] % 2 == 1
i_center_lat = int((lat17600.shape[0] - 1) / 2)
j_center_lon = int((lon17600.shape[1] - 1) / 2)

In [4]:
grp_general = 'MISR_AM1_GRP_ELLIPSOID_GM_P{:d}_O{:d}_{}_F03_0024.hdf'
nadir_file = database / grp_general.format(path, orbit, 'AN')
lat275, lon275 = region.snap_to_grid(path, 275).create_latlon()
brf = mtk.MtkFile(str(nadir_file)).grid('RedBand').field('Red Brf').read(region).data()

In [5]:
tc_cloud_file = database / tc_cloud_general.format(path, orbit)
cmv_grid = mtk.MtkFile(str(tc_cloud_file)).grid('Motion_17.6_km')
CTH = cmv_grid.field('CloudTopHeightOfMotion').read(region).data()
CTH[CTH == -9999] = np.nan
CMV_N = cmv_grid.field('CloudMotionNorthward').read(region).data()
CMV_N[CMV_N == -9999] = np.nan
CMV_E = cmv_grid.field('CloudMotionEastward').read(region).data()
CMV_E[CMV_E == -9999] = np.nan
CMV_Speed = (CMV_N**2 + CMV_E**2)**0.5

In [6]:
cmv_data = xr.Dataset(
    data_vars = {
        'cth_motion': (['y17600', 'x17600'], CTH),
        'cmv_N': (['y17600', 'x17600'], CMV_N),
        'cmv_E': (['y17600', 'x17600'], CMV_E),
        'lat17600': (['y17600', 'x17600'], lat17600),
        'lon17600': (['y17600', 'x17600'], lon17600)
    },
    attrs = {
        'orbit': orbit,
        'path': path,
        'blocks': region.block_range(path),
    }
)
cmv_data['cth_motion'].attrs['units'] = 'm'
cmv_data['cth_motion'].attrs['long_name'] = 'Cloud Top Height Of Motion'
cmv_data['cmv_N'].attrs['units'] = 'm/s'
cmv_data['cmv_N'].attrs['long_name'] = 'Cloud Motion Northward'
cmv_data['cmv_E'].attrs['units'] = 'm/s'
cmv_data['cmv_E'].attrs['long_name'] = 'Cloud Motion Eastward'
cmv_data.to_netcdf(out_file)

In [7]:
# f, ax = plt.subplots(1, 1, figsize=(8, 6))
# ax.scatter(lon275, lat275, c=brf, marker='s', s=1, cmap='gray', vmin=0, vmax=0.3)
# q = ax.quiver(lon17600, lat17600, CMV_E, CMV_N, CTH, units='xy', clim=(0, 1200))
# ax.quiverkey(q, 0.82, 0.06, 5, '5 m/s', labelpos='N', coordinates='figure')
# plt.colorbar(q, label='Cloud Top Height (m)')
# ax.set_xlabel('Longitude (deg)')
# ax.set_ylabel('Latitude (deg)')

In [8]:
# center_CTH = CTH[i_center_lat, j_center_lon]
# half_bin = 25 # m
# cond = (CTH > (center_CTH - half_bin)) & (CTH < (center_CTH + half_bin))
# f, ax = plt.subplots(1, 1, figsize=(8, 6))
# ax.scatter(lon275, lat275, c=brf, marker='s', s=1, cmap='gray')
# q = ax.quiver(lon17600[cond], lat17600[cond], CMV_E[cond], CMV_N[cond], CTH[cond], units='xy')
# ax.quiverkey(q, 0.82, 0.06, 5, '5 m/s', labelpos='N', coordinates='figure')
# plt.colorbar(q, label='Cloud Top Height (m)')
# ax.set_xlabel('Longitude (deg)')
# ax.set_ylabel('Latitude (deg)')

In [9]:
# print('Center CTH: {:.0f} m'.format(CTH[i_center_lat, j_center_lon]))
# print('Center CMV East: {:.2f} m/s'.format(CMV_E[i_center_lat, j_center_lon]))
# print('Center CMV North: {:.2f} m/s'.format(CMV_N[i_center_lat, j_center_lon]))
# print('Center CMV Speed: {:.2f} m/s'.format(CMV_Speed[i_center_lat, j_center_lon]))
# print('Center CMV Direction: {:.0f} deg'.format(np.arctan2(CMV_E[i_center_lat, j_center_lon], CMV_N[i_center_lat, j_center_lon]) * 180 / np.pi))
# print()
# print('Average of n = {:d} CMVs between {:.0f} and {:.0f} m:'.format(np.sum(cond), center_CTH - half_bin, center_CTH + half_bin))
# print('Average CMV East: {:.2f} m/s'.format(np.mean(CMV_E[cond])))
# print('Average CMV North: {:.2f} m/s'.format(np.mean(CMV_N[cond])))
# print('Average CMV Speed: {:.2f} m/s'.format((np.mean(CMV_E[cond])**2 + np.mean(CMV_N[cond])**2)**0.5))
# print('Average CMV Direction: {:.0f} deg'.format(np.arctan2(np.mean(CMV_E[cond]), np.mean(CMV_N[cond])) * 180 / np.pi))
# print()
# print('Average CMV Speed: {:.2f} m/s'.format(np.mean(CMV_Speed[cond])))